# Facility-level heat demand vs country-level useful energy demand

Compares the **bottom-up facility heat demand** (`facilities_2024_eu_deduplicated.csv`,
METHODS §3–4, duplicates removed) with the **country-level industry useful energy
demand** in `References/industry_useful_energy_demand/` (per-country CSVs, TWh,
7 energy-demand types × 12 sectors).

**Comparison basis** — reference heat = the four *Non-electric process heat* bands
+ *Steam (non-electric boilers)*. Both electricity rows are excluded, matching the
bottom-up which is fuel-based heat only (METHODS §6).

Scope caveats to keep in mind:

- **Concept**: the reference is *useful energy demand*; our estimate is *useful heat*
  (fuel × boiler efficiency for steam sectors, fuel = heat for direct-fired kilns).
  Close, but not identical — kiln-sector coverage ratios read slightly high.
- **This reference is a different yardstick from the Eurostat top-down** used in
  `analyze_heat_demand.ipynb`: e.g. steel here is ~100 TWh EU-wide vs ~272 TWh
  (Eurostat final energy × heat share), food ~103 vs ~192 TWh. Coverage numbers are
  therefore *not* comparable between the two notebooks.
- **GBR is excluded** (no reference file; ~41 TWh of facility heat).
- Reference sectors with **no facility coverage** (Transport Equipment, Machinery,
  Wood) are left out of the comparison.
- **Food is NOT an independent comparison in this notebook**: since the food
  country-rescaling (METHODS §3), each country's food heat is *scaled to this very
  reference*, so its coverage is ~100% by construction (94–99% after duplicate removal).
- **Taxonomy joints**: lime is grouped with cement here (the reference has no lime
  column; some of it may live in their Ceramics & Glass); our glass-only estimate is
  compared against their Ceramics **&** Glass, which includes ceramics/bricks we
  don't model — expect an undershoot there.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().parent.parent if Path.cwd().name == "analysis_outputs" else Path.cwd()
FAC = ROOT / "heat_demand/facilities/facilities_2024_eu_deduplicated.csv"
REF_DIR = ROOT / "References/industry_useful_energy_demand"
OUT = ROOT / "heat_demand/analysis_outputs/heat_by_country_comparison.csv"

# dataviz reference palette (validated, fixed slot order)
C_BLUE, C_AQUA, C_RED = "#2a78d6", "#1baf7a", "#e34948"
CAT8 = ["#2a78d6", "#1baf7a", "#eda100", "#008300", "#4a3aa7", "#e34948", "#e87ba4", "#eb6834"]

ISO3_TO_EU2 = {
    "AUT": "AT", "BEL": "BE", "BGR": "BG", "CYP": "CY", "CZE": "CZ", "DEU": "DE",
    "DNK": "DK", "ESP": "ES", "EST": "EE", "FIN": "FI", "FRA": "FR", "GRC": "EL",
    "HRV": "HR", "HUN": "HU", "IRL": "IE", "ITA": "IT", "LTU": "LT", "LUX": "LU",
    "LVA": "LV", "NLD": "NL", "POL": "PL", "PRT": "PT", "ROU": "RO", "SVK": "SK",
    "SVN": "SI", "SWE": "SE",
}  # GBR has no reference file

SECTOR_MAP = {
    "iron-and-steel": "Steel",
    "chemicals": "Chemicals", "petrochemical-steam-cracking": "Chemicals",
    "cement": "Cement (+lime)", "lime": "Cement (+lime)",
    "glass": "Ceramics & Glass",
    "pulp-and-paper": "Pulp & Paper",
    "food-beverage-tobacco": "Food, Bev & Tobacco",
    "textiles-leather-apparel": "Textiles & Leather",
    "aluminum": "Non-ferrous Metals", "other-metals": "Non-ferrous Metals",
}

REF_COLS = {
    "Primary Steel (TWh)": "Steel", "Secondary Steel (TWh)": "Steel",
    "Chemicals (TWh)": "Chemicals", "Cement (TWh)": "Cement (+lime)",
    "Pulp and Paper (TWh)": "Pulp & Paper",
    "Food, Beverages and Tobacco (TWh)": "Food, Bev & Tobacco",
    "Textiles and Leather (TWh)": "Textiles & Leather",
    "Non-ferrous Metals (TWh)": "Non-ferrous Metals",
    "Ceramics and Glass (TWh)": "Ceramics & Glass",
}
HEAT_ROWS = [
    "Non-electric process heat (<100 C)", "Non-electric process heat (100-400 C)",
    "Non-electric process heat (400-1000 C)", "Non-electric process heat (>1000 C)",
    "Steam (non-electric boilers)",
]

fac = pd.read_csv(FAC)
fac["cc"] = fac["iso3_country"].map(ISO3_TO_EU2)
fac["sector"] = fac["subsector"].map(SECTOR_MAP)
bu = (
    fac.dropna(subset=["cc"])
    .groupby(["cc", "sector"])["useful_heat_tj"].sum()
    .div(3600).rename("bottom_up_twh").reset_index()
)

refs = []
for f in sorted(REF_DIR.glob("industry_useful_energy_demand_*.csv")):
    cc = f.stem.rsplit("_", 1)[1]
    d = pd.read_csv(f)
    heat = d[d["energy_demand_type"].isin(HEAT_ROWS)]
    s = heat[list(REF_COLS)].sum().rename(index=REF_COLS).groupby(level=0).sum()
    s.index.name = "sector"
    refs.append(s.rename("reference_twh").reset_index().assign(cc=cc))
ref = pd.concat(refs, ignore_index=True)
eu27 = ref[ref["cc"] == "EU27"].set_index("sector")["reference_twh"]
ref = ref[ref["cc"] != "EU27"]

cmp = ref.merge(bu, on=["cc", "sector"], how="left").fillna({"bottom_up_twh": 0})
cmp["coverage_pct"] = 100 * cmp["bottom_up_twh"] / cmp["reference_twh"]
cmp.to_csv(OUT, index=False)

totals = cmp.groupby("cc")[["bottom_up_twh", "reference_twh"]].sum()
totals["coverage_pct"] = 100 * totals["bottom_up_twh"] / totals["reference_twh"]
print(f"EU-26 covered sectors: bottom-up {totals.bottom_up_twh.sum():.0f} TWh "
      f"vs reference {totals.reference_twh.sum():.0f} TWh "
      f"({100 * totals.bottom_up_twh.sum() / totals.reference_twh.sum():.0f}%)")
totals.sort_values("reference_twh", ascending=False).round(1)

EU-26 covered sectors: bottom-up 598 TWh vs reference 794 TWh (75%)


,bottom_up_twh,reference_twh,coverage_pct
cc,,,
DE,133.0,176.2,75.5
IT,61.3,86.2,71.1
FR,66.4,77.3,85.9
ES,54.7,64.5,84.7
NL,40.6,59.6,68.1
PL,38.3,49.4,77.5
FI,24.3,45.8,53.1
SE,22.7,44.1,51.6
BE,29.3,39.7,73.8


## Country totals — bottom-up vs reference

Covered sectors only (steel, chemicals, cement+lime, ceramics & glass, pulp & paper,
food, textiles, non-ferrous). Sorted by reference demand.

In [2]:
t = totals.sort_values("reference_twh").reset_index()
fig = go.Figure()
fig.add_bar(y=t["cc"], x=t["reference_twh"], name="Reference (useful energy)",
            orientation="h", marker_color=C_AQUA,
            hovertemplate="%{y}: %{x:.1f} TWh (reference)<extra></extra>")
fig.add_bar(y=t["cc"], x=t["bottom_up_twh"], name="Bottom-up (facilities)",
            orientation="h", marker_color=C_BLUE,
            hovertemplate="%{y}: %{x:.1f} TWh (bottom-up)<extra></extra>")
fig.update_layout(
    title="Industrial process heat by country: facility bottom-up vs reference",
    barmode="group", template="plotly_white", height=720, width=850,
    xaxis_title="Heat demand (TWh_th/yr)", yaxis_title=None,
    legend=dict(orientation="h", y=1.02, x=0),
)
fig.show()

## Where the gaps are — coverage ratio by country x sector

Bottom-up / reference, % (100 = perfect agreement). Blue = bottom-up under the
reference, red = over. Cells are capped at 250% for readability; hover shows the
exact value. Countries ordered by reference demand, sectors by EU coverage.

In [3]:
order_cc = totals.sort_values("reference_twh", ascending=False).index
sec_tot = cmp.groupby("sector")[["bottom_up_twh", "reference_twh"]].sum()
order_sec = (100 * sec_tot.bottom_up_twh / sec_tot.reference_twh).sort_values().index

grid = cmp.pivot(index="sector", columns="cc", values="coverage_pct")
grid = grid.reindex(index=order_sec, columns=order_cc)
ref_grid = cmp.pivot(index="sector", columns="cc", values="reference_twh").reindex(
    index=order_sec, columns=order_cc)
# blank out cells where the reference itself is negligible (<0.1 TWh)
show = grid.where(ref_grid >= 0.1)

fig = go.Figure(go.Heatmap(
    z=show.clip(upper=250).values, x=show.columns, y=show.index,
    zmin=0, zmax=250, zmid=100,
    colorscale=[[0.0, "#0d366b"], [0.2, "#2a78d6"], [0.4, "#f0f0ee"],
                [0.53, "#e34948"], [1.0, "#7f1d1d"]],
    customdata=np.dstack([np.where(np.isnan(show.values), 0, show.values),
                          np.where(np.isnan(ref_grid.values), 0, ref_grid.values)]),
    hovertemplate="%{x} - %{y}<br>coverage %{customdata[0]:.0f}%"
                  "<br>reference %{customdata[1]:.1f} TWh<extra></extra>",
    colorbar=dict(title="coverage %", tickvals=[0, 50, 100, 150, 200, 250],
                  ticktext=["0", "50", "100", "150", "200", "250+"]),
))
fig.update_layout(title="Coverage of reference heat demand (%), by country and sector",
                  template="plotly_white", height=420, width=980)
fig.show()

## Country x sector scatter

Each point is one country-sector cell (log-log; points on the diagonal agree with
the reference). Cells below 0.1 TWh reference are dropped.

In [4]:
sc = cmp[(cmp.reference_twh >= 0.1) & (cmp.bottom_up_twh > 0)].copy()
lim = [0.08, 120]
fig = px.scatter(
    sc, x="reference_twh", y="bottom_up_twh", color="sector", hover_name="cc",
    log_x=True, log_y=True, color_discrete_sequence=CAT8,
    labels={"reference_twh": "Reference useful energy (TWh)",
            "bottom_up_twh": "Facility bottom-up heat (TWh)"},
    hover_data={"coverage_pct": ":.0f"},
)
fig.add_shape(type="line", x0=lim[0], y0=lim[0], x1=lim[1], y1=lim[1],
              line=dict(color="#9a9a94", dash="dot", width=1))
fig.update_traces(marker=dict(size=9, line=dict(width=1, color="white")))
fig.update_layout(title="Country x sector: bottom-up vs reference (1:1 dotted)",
                  template="plotly_white", height=560, width=880,
                  xaxis_range=np.log10(lim), yaxis_range=np.log10(lim))
fig.show()

## EU totals by sector — and how this reference differs from the Eurostat top-down

In [5]:
sec = cmp.groupby("sector")[["bottom_up_twh", "reference_twh"]].sum()
sec["eu27_file_twh"] = eu27
sec["coverage_pct"] = (100 * sec.bottom_up_twh / sec.reference_twh).round(1)
s = sec.sort_values("reference_twh").reset_index()
fig = go.Figure()
fig.add_bar(y=s["sector"], x=s["reference_twh"], name="Reference (useful energy)",
            orientation="h", marker_color=C_AQUA,
            hovertemplate="%{y}: %{x:.0f} TWh<extra></extra>")
fig.add_bar(y=s["sector"], x=s["bottom_up_twh"], name="Bottom-up (facilities)",
            orientation="h", marker_color=C_BLUE,
            hovertemplate="%{y}: %{x:.0f} TWh<extra></extra>")
fig.update_layout(title="EU-26 heat demand by sector: bottom-up vs reference",
                  barmode="group", template="plotly_white", height=420, width=850,
                  xaxis_title="Heat demand (TWh_th/yr)",
                  legend=dict(orientation="h", y=1.02, x=0))
fig.show()
sec.round(1)

,bottom_up_twh,reference_twh,eu27_file_twh,coverage_pct
sector,,,,
Cement (+lime),123.6,75.8,75.8,163.0
Ceramics & Glass,34.4,106.1,106.1,32.4
Chemicals,120.9,172.5,172.5,70.1
"Food, Bev & Tobacco",100.6,103.0,103.0,97.7
Non-ferrous Metals,10.5,22.2,22.2,47.1
Pulp & Paper,54.1,203.4,203.4,26.6
Steel,138.7,99.5,99.5,139.4
Textiles & Leather,15.1,11.5,11.5,130.7


## Country x temperature band

Same comparison, cut by temperature band instead of sector. Band mapping
(edges do not align exactly - see caveats):

| Comparison band | Bottom-up (hotmaps bands) | Reference rows |
|---|---|---|
| <100 C | <100 C | Non-electric process heat (<100 C) |
| 100-400 C | 100-200 C + 200-500 C | Non-electric (100-400 C) + **Steam (non-electric boilers)** |
| 400-1000 C | 500-1000 C | Non-electric (400-1000 C) |
| >1000 C | >1000 C | Non-electric (>1000 C) |

Caveats: our 200-500 C band straddles their 400 C edge (small - ~6% of EU heat);
steam is assigned to 100-400 C (industrial steam is mostly 120-250 C); reference
restricted to the covered sector columns. **Band-edge conventions differ for
high-temperature kilns**: the hotmaps benchmarks put cement calcination (~900 C)
and cracker furnaces in 500-1000 C, while the reference books kiln heat >1000 C -
so those two bands should be read together (the ">400 C combined" column).

In [6]:
BAND_COLS = [f"heat_{x}_tj" for x in
             ["below100C", "100C-200C", "200C-500C", "500C-1000C", "above1000C"]]
BANDS4 = ["<100C", "100-400C", "400-1000C", ">1000C"]
BAND_RAMP = {"<100C": "#86b6ef", "100-400C": "#3987e5",
             "400-1000C": "#1c5cab", ">1000C": "#104281"}

# bottom-up: rescale band columns onto the useful_heat basis, then coarsen
bb = fac[fac["bench_heat_tj"].gt(0) & fac["useful_heat_tj"].notna()
         & fac["cc"].notna()].copy()
_sh = bb[BAND_COLS].div(bb[BAND_COLS].sum(axis=1), axis=0)
for _c in BAND_COLS:
    bb[_c] = _sh[_c] * bb["useful_heat_tj"]
bu_band = pd.DataFrame({
    "<100C": bb.groupby("cc")[BAND_COLS[0]].sum(),
    "100-400C": bb.groupby("cc")[BAND_COLS[1]].sum()
                + bb.groupby("cc")[BAND_COLS[2]].sum(),
    "400-1000C": bb.groupby("cc")[BAND_COLS[3]].sum(),
    ">1000C": bb.groupby("cc")[BAND_COLS[4]].sum(),
}) / 3600

# reference: band rows x covered sector columns
ROWMAP = {"Non-electric process heat (<100 C)": "<100C",
          "Non-electric process heat (100-400 C)": "100-400C",
          "Steam (non-electric boilers)": "100-400C",
          "Non-electric process heat (400-1000 C)": "400-1000C",
          "Non-electric process heat (>1000 C)": ">1000C"}
ref_rows = {}
for f in sorted(REF_DIR.glob("industry_useful_energy_demand_*.csv")):
    cc = f.stem.rsplit("_", 1)[1]
    if cc == "EU27":
        continue
    d = pd.read_csv(f)
    d["band"] = d["energy_demand_type"].map(ROWMAP)
    ref_rows[cc] = (d.dropna(subset=["band"])
                    .groupby("band")[list(REF_COLS)].sum().sum(axis=1))
ref_band = pd.DataFrame(ref_rows).T[BANDS4]
bu_band = bu_band.reindex(ref_band.index).fillna(0)

band_cmp = pd.concat(
    {"bottom_up_twh": bu_band, "reference_twh": ref_band}, axis=1
).round(3)
band_cmp.to_csv(ROOT / "heat_demand/analysis_outputs/heat_by_country_band_comparison.csv")

eu_b, eu_r = bu_band.sum(), ref_band.sum()
for bd in BANDS4:
    print(f"{bd:>10}: {eu_b[bd]:6.0f} vs {eu_r[bd]:6.0f} TWh  ({100*eu_b[bd]/eu_r[bd]:3.0f}%)")
print(f"   >400C combined: {eu_b[BANDS4[2:]].sum():.0f} vs {eu_r[BANDS4[2:]].sum():.0f} TWh"
      f"  ({100*eu_b[BANDS4[2:]].sum()/eu_r[BANDS4[2:]].sum():.0f}%)")

     <100C:     64 vs    121 TWh  ( 53%)
  100-400C:    156 vs    362 TWh  ( 43%)
 400-1000C:    246 vs     79 TWh  (311%)
    >1000C:    131 vs    232 TWh  ( 57%)
   >400C combined: 378 vs 311 TWh  (121%)


In [7]:
# coverage heatmap, incl. the >400C combined column
cov = 100 * bu_band / ref_band
cov[">400C (combined)"] = 100 * (bu_band["400-1000C"] + bu_band[">1000C"]) / (
    ref_band["400-1000C"] + ref_band[">1000C"])
ref_show = ref_band.copy()
ref_show[">400C (combined)"] = ref_band["400-1000C"] + ref_band[">1000C"]
order_cc2 = ref_band.sum(axis=1).sort_values(ascending=False).index
covT = cov.reindex(order_cc2).T.where(ref_show.reindex(order_cc2).T >= 0.1)

fig = go.Figure(go.Heatmap(
    z=covT.clip(upper=250).values, x=covT.columns, y=covT.index,
    zmin=0, zmax=250, zmid=100,
    colorscale=[[0.0, "#0d366b"], [0.2, "#2a78d6"], [0.4, "#f0f0ee"],
                [0.53, "#e34948"], [1.0, "#7f1d1d"]],
    customdata=np.dstack([np.nan_to_num(covT.values),
                          np.nan_to_num(ref_show.reindex(order_cc2).T.values)]),
    hovertemplate="%{x} - %{y}<br>coverage %{customdata[0]:.0f}%"
                  "<br>reference %{customdata[1]:.1f} TWh<extra></extra>",
    colorbar=dict(title="coverage %", tickvals=[0, 50, 100, 150, 200, 250],
                  ticktext=["0", "50", "100", "150", "200", "250+"]),
))
fig.update_layout(title="Coverage of reference heat by temperature band (%), by country",
                  template="plotly_white", height=360, width=980)
fig.show()

In [8]:
# top-10 countries: paired stacked bars (bottom-up | reference) by band
top10 = list(ref_band.sum(axis=1).sort_values(ascending=False).head(10).index)
fig = go.Figure()
for est, data in [("Bottom-up", bu_band), ("Reference", ref_band)]:
    for bd in BANDS4:
        fig.add_bar(
            x=[[c for c in top10], [est] * len(top10)],
            y=data.loc[top10, bd], name=bd, legendgroup=bd,
            showlegend=(est == "Bottom-up"), marker_color=BAND_RAMP[bd],
            marker_line=dict(color="#fcfcfb", width=1),
            hovertemplate=f"%{{x}} {bd}: %{{y:.1f}} TWh<extra>{est}</extra>",
        )
fig.update_layout(
    barmode="stack", template="plotly_white", height=480, width=1000,
    title="Heat by temperature band - bottom-up vs reference, top 10 countries",
    yaxis_title="Heat demand (TWh_th/yr)",
    legend=dict(orientation="h", y=1.02, x=0),
)
fig.show()

## Findings

**Overall, the facility bottom-up reproduces the country picture well**: EU-26
covered-sector total is 598 TWh vs 794 TWh reference (75%), and the largest
industrial countries all land in a plausible band — DE 75%, FR 86%, ES 85%,
IT 71%, PL 77%.

**Country outliers mirror the facility-level caveats (METHODS §6):**

- **Portugal is fixed (now 94% overall)** — its Climate TRACE food emissions were
  ~18x overstated (22.7 TWh bottom-up vs 1.6 TWh reference before the fix); the food
  country-rescaling (METHODS §3) now pins every country's food total to this
  reference, using Climate TRACE only for within-country site allocation. Three PT
  cement works double-listed as Mt-scale lime plants were also found via this
  comparison and removed (duplicate_sites_flagged.csv XS-3..XS-9, which caught the
  same pattern in LT, ES, FR/LU, CZ and the closed Halyvourgiki steel plant in EL).
- **Finland & Sweden ~51%** — pulp-&-paper-dominated countries where the
  bottom-up counts only ~27% of reference P&P heat EU-wide. Our 10.5 GJ/t SEC is
  total heat per tonne, but the reference books the full steam demand
  (~203 TWh EU) regardless of whether black liquor supplies it; the coverage gap
  is mostly the biomass-supplied share we deliberately down-weight.
- **Ireland 138%, Greece 133%, Bulgaria 123%** — small totals, dominated by
  modeled food/textile emissions (tier 3), so ratios are noisy.

**Sector-level, this reference is a different yardstick from the Eurostat
top-down in `analyze_heat_demand.ipynb`** — the two should not be mixed:

| sector | vs Eurostat x heat-share | vs this reference |
|---|---|---|
| Steel | 54% of 272 TWh | 139% of 100 TWh |
| Food | 58% of 192 TWh | ~100% (by construction) |
| Pulp & paper | 21% of 260 TWh | 27% of 203 TWh |

Steel sits *between* the two references: the Eurostat number is final energy x
heat share (upper bound), while this useful-energy reference nets out conversion
losses (lower bound). Food is anchored to this reference by the country
rescaling. Our estimates use eff = 1.0 for direct-fired
kilns (fuel = heat by convention), which reads high against a useful-energy
basis — cement (+lime) at 170% is largely this convention plus the lime/ceramics
taxonomy joint; Ceramics & Glass at 32% is the mirror image (we model glass
only, not ceramics/bricks).

**Bottom line:** country-level agreement is strongest exactly where the facility
data is strongest (tonnage sectors in large countries), and every large
country-level discrepancy traces back to a known facility-level issue rather
than to new, unexplained error — which is the consistency the facility-level
crosschecks hoped for.

**By temperature band** (EU-26): <100 C at 53%, 100-400 C at 43%, 400-1000 C at
~310%, >1000 C at 57% - but the two high bands are a classification clash, not a
magnitude error: **>400 C combined is 121%**, consistent with the fuel-vs-useful
convention for kilns. The hotmaps benchmarks put cement calcination (~900 C) and
cracker furnaces in 500-1000 C while the reference books kiln heat >1000 C; read
those bands together. The 100-400 C shortfall (43%) tracks the sector coverage
gaps - the reference's big boiler-steam demand sits in chemicals and paper, our
two weakest sectors. Finland/Sweden show ~3% coverage below 100 C because the
reference books substantial low-temperature paper-sector heat there while our
pulp recipes put mill heat almost entirely in 100-200 C.